In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import json
import os
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from datetime import datetime, timezone
from sgp4.api import Satrec

print("Imports loaded!")

In [ ]:
TLE_CACHE_FILE   = os.path.expanduser("~/tle_cache.json")
SATCAT_CACHE_FILE = os.path.expanduser("~/satcat_cache.json")

# Load TLE cache
with open(TLE_CACHE_FILE) as f:
    tle_cache = json.load(f)

# Load SATCAT cache
with open(SATCAT_CACHE_FILE) as f:
    satcat_cache = json.load(f)

tles       = tle_cache['tles']
tle_age    = datetime.now(timezone.utc) - datetime.fromisoformat(tle_cache['fetched_at'])
satcat_age = datetime.now(timezone.utc) - datetime.fromisoformat(satcat_cache['fetched_at'])

print(f"Caches loaded!")
print(f"   Launch:           {satcat_cache['intldes']}")
print(f"   SATCAT age:       {int(satcat_age.total_seconds()//60)} min")
print(f"   TLE age:          {int(tle_age.total_seconds()//60)} min")
print(f"   Total objects:    {len(satcat_cache['norad_ids'])}")
print(f"   On-orbit objects: {len(tles)}")

In [ ]:
# Ground Station — Cal Poly SLO
gs_name = "Cal Poly SLO"
gs_lat  = 35.30    # degrees North
gs_lon  = -120.66  # degrees East
gs_alt  = 0.097    # km

gs_lat_rad = np.radians(gs_lat)
gs_lon_rad = np.radians(gs_lon)
gs_location = EarthLocation(lat=gs_lat*u.deg, lon=gs_lon*u.deg, height=gs_alt*u.km)

# Propagation parameters
now        = Time(datetime.now(timezone.utc))
duration_s = 24 * 3600  # 24 hours
num_steps  = 2000
min_elev   = 10.0  # degrees

times = now + np.linspace(0, duration_s, num_steps) * u.s

print(f"Ground station: {gs_name}")
print(f"   Lat: {gs_lat}° Lon: {gs_lon}° Alt: {gs_alt} km")
print(f"   Start: {now.iso} UTC")
print(f"   Steps: {num_steps} over {duration_s/3600:.0f} hours")
print(f"   Min elevation: {min_elev}°")

In [ ]:
from sgp4.api import SatrecArray

#Build SatrecArray from all TLEs
satellites = SatrecArray([
    Satrec.twoline2rv(t['line1'], t['line2']) for t in tles
])

# Vectorized propagation - all satellites, all timesteps in one call
jd1 = np.array([t.jd1 for t in times])
jd2 = np.array([t.jd2 for t in times])

# Shape: (num_satellites, num_steps, 3)
e, r_teme, v_teme = satellites.sgp4(jd1, jd2)

error_counts = (e != 0).sum(axis=1)  # errors per satellite
for i, count in enumerate(error_counts):
    if count > 0:
        print(f"   {tles[i]['name']}: {count} error timesteps")

print(f"Propagation complete!")
print(f"   Satellites: {r_teme.shape[0]}")
print(f"   Timesteps:  {r_teme.shape[1]}")
print(f"   Errors:     {(e != 0).sum()} non-zero error codes")

In [ ]:
# Convert TEME to lat/lon/alt for all satellites
lats = np.zeros((len(tles), num_steps))
lons = np.zeros((len(tles), num_steps))
alts = np.zeros((len(tles), num_steps))

for i in range(len(tles)):
    for j in range(num_steps):
        if e[i, j] != 0:
            continue
        r = CartesianRepresentation(r_teme[i, j] * u.km)
        teme_coord = TEME(r, obstime=times[j])
        itrs = teme_coord.transform_to(ITRS(obstime=times[j]))
        loc = EarthLocation(*itrs.cartesian.xyz)
        lats[i, j] = loc.lat.deg
        lons[i, j] = loc.lon.deg
        alts[i, j] = loc.height.to(u.km).value

print(f"Conversion complete!")
print(f"\n{'Name':<25} {'Min Alt':>8} {'Max Alt':>8} {'Mean Alt':>8}")
print("-" * 55)
for i, t in enumerate(tles):
    valid = alts[i][e[i] == 0]
    if len(valid) > 0:
        print(f"{t['name']:<25} {valid.min():>8.1f} {valid.max():>8.1f} {valid.mean():>8.1f}")

In [ ]:
import numpy as np

PROP_CACHE_FILE = os.path.expanduser("~/propagation_cache.npz")

np.savez(PROP_CACHE_FILE,
    r_teme     = r_teme,
    v_teme     = v_teme,
    e          = e,
    lats       = lats,
    lons       = lons,
    alts       = alts,
    times_jd1  = np.array([t.jd1 for t in times]),
    times_jd2  = np.array([t.jd2 for t in times])
)

print(f"Propagation cached!")
print(f"   Shape: {r_teme.shape} — (satellites, timesteps, xyz)")
print(f"   File:  {PROP_CACHE_FILE}")